# 주성분분석 실습

**PCA · Principal Component Analysis**

데이터 분산이 큰 방향을 찾아 적은 수의 축으로 데이터를 다시 표현하는 선형 차원축소.

소재 분야에서 이해하기: 수십 개 기술자를 두 축으로 줄여 조성 분포를 살펴본다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 분해 기법 문서](https://scikit-learn.org/stable/modules/decomposition.html)

## 1. 상관된 기술자를 몇 축으로 줄이기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n = 300
base = rng.normal(0, 1, (n, 3))
descriptors = np.column_stack([
    base[:, 0], base[:, 0] * 0.95 + rng.normal(0, 0.1, n),   # 거의 같은 정보
    base[:, 1], base[:, 1] * -0.9 + rng.normal(0, 0.15, n),
    base[:, 2], rng.normal(0, 1, n)])                        # 마지막은 잡음
print('기술자 %d개' % descriptors.shape[1])

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(descriptors)
pca = PCA().fit(scaled)
ratio = pca.explained_variance_ratio_
for index, value in enumerate(ratio, 1):
    print('주성분 %d: 분산 설명 %.1f%% (누적 %.1f%%)' % (index, 100 * value, 100 * ratio[:index].sum()))
plt.plot(range(1, len(ratio) + 1), np.cumsum(ratio), 'o-')
plt.axhline(0.9, color='r', ls='--')
plt.xlabel('number of components'); plt.ylabel('cumulative explained variance'); plt.show()

In [ ]:
projected = PCA(n_components=2).fit_transform(scaled)
plt.scatter(projected[:, 0], projected[:, 1], s=14, c=base[:, 0], cmap='viridis')
plt.colorbar(label='hidden factor 1')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()
print('첫 두 축만으로 숨은 인자가 색 순서대로 배열되는지 확인하세요.')

## 2. 해석

PCA는 분산이 큰 방향을 찾습니다. 단위가 다른 기술자는 반드시 표준화해야 하고, 분산이 크다는 것이
예측에 중요하다는 뜻은 아닙니다. 회귀에 쓸 축은 목표값과의 관계로 다시 확인해야 합니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#pca)을 여세요.